In [200]:
# Biblioteker
import pandas as pd
import numpy as np
import statsmodels.api as sm
from linearmodels.panel import PanelOLS, RandomEffects
from scipy.stats import chi2

In [201]:
# Indlæs data
gdp = pd.read_csv("GDP.csv", skiprows=4)
aging = pd.read_csv("age-dependency-ratio-old.csv")
pwt = pd.read_excel("Human-Productivity.xlsx", sheet_name="Data")

In [202]:
# Tjek de har læst korrekt - Kan sletts, kun til kontrol
print(gdp.head())
print(gdp.columns[:10])

print(aging.head())
print(aging.columns)

print(pwt.head())
print(pwt.columns[:15])

                  Country Name Country Code  \
0                        Aruba          ABW   
1  Africa Eastern and Southern          AFE   
2                  Afghanistan          AFG   
3   Africa Western and Central          AFW   
4                       Angola          AGO   

                                  Indicator Name     Indicator Code  1960  \
0  GDP per capita, PPP (current international $)  NY.GDP.PCAP.PP.CD   NaN   
1  GDP per capita, PPP (current international $)  NY.GDP.PCAP.PP.CD   NaN   
2  GDP per capita, PPP (current international $)  NY.GDP.PCAP.PP.CD   NaN   
3  GDP per capita, PPP (current international $)  NY.GDP.PCAP.PP.CD   NaN   
4  GDP per capita, PPP (current international $)  NY.GDP.PCAP.PP.CD   NaN   

   1961  1962  1963  1964  1965  ...          2017          2018  \
0   NaN   NaN   NaN   NaN   NaN  ...  37524.914920  39287.059713   
1   NaN   NaN   NaN   NaN   NaN  ...   3837.726375   3723.216423   
2   NaN   NaN   NaN   NaN   NaN  ...   2335.795862

In [203]:
# Clean GDP data
gdp = gdp.loc[:, ~gdp.columns.str.contains("^Unnamed")]

# Behold kun det nødvendige
gdp = gdp[["Country Name", "Country Code"] + [col for col in gdp.columns if col.isdigit()]]

# Lav fra wide til long format
gdp = gdp.melt(
    id_vars=["Country Name", "Country Code"],
    var_name="Year",
    value_name="GDP"
)

# Omdøb kolonner
gdp = gdp.rename(columns={"Country Name": "Country"})

# Ens datatype
gdp["Year"] = pd.to_numeric(gdp["Year"], errors="coerce")

# Tjek - kan slettes, kun til kontrol
print(gdp.head())
print(gdp.columns)
print(gdp.shape)

                       Country Country Code  Year  GDP
0                        Aruba          ABW  1960  NaN
1  Africa Eastern and Southern          AFE  1960  NaN
2                  Afghanistan          AFG  1960  NaN
3   Africa Western and Central          AFW  1960  NaN
4                       Angola          AGO  1960  NaN
Index(['Country', 'Country Code', 'Year', 'GDP'], dtype='object')
(17556, 4)


In [204]:
# Clean aging
aging = aging.rename(columns={
    "Entity": "Country",
    "Age dependency ratio, old (% of working-age population)": "Age_dependency_old"
})

# Behold kun relevante kolonner
aging = aging[["Country", "Year", "Age_dependency_old"]]

# Ens datatype
aging["Year"] = pd.to_numeric(aging["Year"], errors="coerce")

# Tjek - kan slettes, kun til kontrol
print(aging.head())
print(aging.columns)
print(aging.shape)

       Country  Year  Age_dependency_old
0  Afghanistan  1950            5.078877
1  Afghanistan  1951            5.100585
2  Afghanistan  1952            5.114399
3  Afghanistan  1953            5.122446
4  Afghanistan  1954            5.126267
Index(['Country', 'Year', 'Age_dependency_old'], dtype='object')
(19388, 3)


In [205]:
# Clean PWT
pwt = pwt.rename(columns={
    "country": "Country",
    "year": "Year"
})

# Behold kun relevante kolonner
pwt = pwt[["Country", "Year", "hc"]]

# Ens datatype
pwt["Year"] = pd.to_numeric(pwt["Year"], errors="coerce")

# Tjek - kan slettes, kun til kontrol
print(pwt.head())
print(pwt.columns)
print(pwt.shape)

  Country  Year  hc
0   Aruba  1950 NaN
1   Aruba  1951 NaN
2   Aruba  1952 NaN
3   Aruba  1953 NaN
4   Aruba  1954 NaN
Index(['Country', 'Year', 'hc'], dtype='object')
(13690, 3)


In [206]:
# Merge data
df = aging.merge(gdp, on=["Country", "Year"], how="inner")
df = df.merge(pwt, on=["Country", "Year"], how="inner")

# kan slettes, kun til kontrol
print(df.shape)
print(df.head())
print(df.isna().sum())

(9728, 6)
   Country  Year  Age_dependency_old Country Code  GDP  hc
0  Albania  1960           10.268960          ALB  NaN NaN
1  Albania  1961           10.132406          ALB  NaN NaN
2  Albania  1962            9.930125          ALB  NaN NaN
3  Albania  1963            9.761269          ALB  NaN NaN
4  Albania  1964            9.618436          ALB  NaN NaN
Country                  0
Year                     0
Age_dependency_old       0
Country Code             0
GDP                   4708
hc                    2437
dtype: int64


In [207]:
# Drop rækker med manglende værdier i de relevante kolonner
df_clean = df.dropna(subset=["GDP", "hc", "Age_dependency_old"])

# kan slettes, kun til kontrol
print(df_clean.shape)
print(df_clean.isna().sum())
print(df_clean.head())

(4142, 6)
Country               0
Year                  0
Age_dependency_old    0
Country Code          0
GDP                   0
hc                    0
dtype: int64
    Country  Year  Age_dependency_old Country Code          GDP        hc
30  Albania  1990            8.680737          ALB  2655.673133  2.516159
31  Albania  1991            8.855720          ALB  1988.640611  2.515733
32  Albania  1992            9.075256          ALB  1899.258869  2.515308
33  Albania  1993            9.350662          ALB  2143.176074  2.514883
34  Albania  1994            9.652499          ALB  2385.279580  2.514457


##  Deskriptiv statistik 

In [208]:
# Deskriptive statistikker
print(df_clean.describe())

              Year  Age_dependency_old            GDP           hc
count  4142.000000         4142.000000    4142.000000  4142.000000
mean   2006.521487           12.678355   17937.801232     2.463254
std       9.802562            8.772110   21558.768121     0.703999
min    1990.000000            1.012210     254.388475     1.029605
25%    1998.000000            5.853011    3411.079175     1.877493
50%    2007.000000            8.222938    9573.997735     2.552312
75%    2015.000000           19.654958   24643.546369     3.031884
max    2023.000000           50.120530  180939.439450     3.986023


In [209]:
# Sortér data
df_clean = df_clean.sort_values(["Country", "Year"])

In [210]:
# Beregn årlig vækst i GDP
df_clean["gdp_growth"] = df_clean.groupby("Country")["GDP"].pct_change()

In [211]:
# Identificer recessioner
df_clean["recession"] = (df_clean["gdp_growth"] < 0).astype(int)

In [212]:
# Drop rækker med manglende værdier i gdp_growth
df_clean = df_clean.dropna(subset=["gdp_growth"])

In [213]:
df_clean[["Country", "Year", "GDP", "gdp_growth", "recession"]].head(10)

,Country,Year,GDP,gdp_growth,recession
31,Albania,1991,1988.640611,-0.251173,1
32,Albania,1992,1899.258869,-0.044946,1
33,Albania,1993,2143.176074,0.128428,0
34,Albania,1994,2385.279580,0.112965,0
35,Albania,1995,2776.828814,0.164152,0
36,Albania,1996,3054.035206,0.099828,0
37,Albania,1997,2760.065364,-0.096256,1
38,Albania,1998,3085.785894,0.118012,0
39,Albania,1999,3549.272560,0.150201,0
40,Albania,2000,3977.771869,0.120729,0


### OLS regression (Ordinary Least Squares)

In [214]:
df_clean["log_gdp"] = np.log(df_clean["GDP"])

X = df_clean[["Age_dependency_old", "hc", "recession"]]
X = sm.add_constant(X)

y = df_clean["log_gdp"]

model = sm.OLS(y, X).fit()
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:                log_gdp   R-squared:                       0.631
Model:                            OLS   Adj. R-squared:                  0.630
Method:                 Least Squares   F-statistic:                     2284.
Date:                Wed, 01 Apr 2026   Prob (F-statistic):               0.00
Time:                        13:54:20   Log-Likelihood:                -4717.9
No. Observations:                4020   AIC:                             9444.
Df Residuals:                    4016   BIC:                             9469.
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                         coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------
const                  5.5887      0

### FE

In [219]:
from linearmodels.panel import PanelOLS
import numpy as np

df_clean["log_gdp"] = np.log(df_clean["GDP"])
df_panel = df_clean.set_index(["Country", "Year"])

fe = PanelOLS.from_formula(
    "log_gdp ~ Age_dependency_old + hc + recession + EntityEffects + TimeEffects",
    data=df_panel
).fit()

print(fe.summary)

                          PanelOLS Estimation Summary                           
Dep. Variable:                log_gdp   R-squared:                        0.0590
Estimator:                   PanelOLS   R-squared (Between):              0.0419
No. Observations:                4020   R-squared (Within):               0.1182
Date:                Wed, Apr 01 2026   R-squared (Overall):              0.0420
Time:                        14:02:51   Log-likelihood                    1041.9
Cov. Estimator:            Unadjusted                                           
                                        F-statistic:                      80.785
Entities:                         122   P-value                           0.0000
Avg Obs:                       32.951   Distribution:                  F(3,3863)
Min Obs:                       28.000                                           
Max Obs:                       33.000   F-statistic (robust):             80.785
                            

In [220]:
from linearmodels.panel import RandomEffects

# (df_panel har du allerede lavet)
# df_panel = df_clean.set_index(["Country", "Year"])

re = RandomEffects.from_formula(
    "log_gdp ~ Age_dependency_old + hc + recession",
    data=df_panel
).fit()

print(re.summary)

                        RandomEffects Estimation Summary                        
Dep. Variable:                log_gdp   R-squared:                        0.7038
Estimator:              RandomEffects   R-squared (Between):              0.7744
No. Observations:                4020   R-squared (Within):               0.6749
Date:                Wed, Apr 01 2026   R-squared (Overall):              0.7740
Time:                        14:09:31   Log-likelihood                   -751.20
Cov. Estimator:            Unadjusted                                           
                                        F-statistic:                      3181.6
Entities:                         122   P-value                           0.0000
Avg Obs:                       32.951   Distribution:                  F(3,4017)
Min Obs:                       28.000                                           
Max Obs:                       33.000   F-statistic (robust):             3181.6
                            

### Hausman test

In [221]:
import numpy as np
from scipy.stats import chi2

# Hent koefficienter (kun fælles variable)
fe_params = fe.params[["Age_dependency_old", "hc", "recession"]]
re_params = re.params[["Age_dependency_old", "hc", "recession"]]

# Covariance matrices
fe_cov = fe.cov.loc[fe_params.index, fe_params.index]
re_cov = re.cov.loc[re_params.index, re_params.index]

# Forskel
diff = fe_params - re_params

# Hausman statistik
stat = np.dot(np.dot(diff.T, np.linalg.inv(fe_cov - re_cov)), diff)

# p-value
df = len(diff)
p_value = 1 - chi2.cdf(stat, df)

print("Hausman test statistic:", stat)
print("p-value:", p_value)

Hausman test statistic: 4909.979958410042
p-value: 0.0
